In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
data = np.loadtxt('stock_volat_curated.csv', delimiter=',')

data = data.T
np.random.shuffle(data)
data = data.T

X = data[:505,:]
Y = data[-1:,:] # Using volatility now

print(X.shape)
print(Y.shape)

In [ ]:
print("Volatility Stats:")
std_Y = Y.std()
print("Standard Deviation:", std_Y)
print("Variance:", std_Y ** 2)
print("Second standard Deviation:", 2 * std_Y)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(505,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(126, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanAbsoluteError(),
    metrics=['mae']
)

model.summary()


In [ ]:
# Clip extreme outliers in Y so MSE isn't dominated by rare spikes
# (p99 ≈ 2.09, but max ≈ 19 — those few outliers monopolize the gradient)
#y_lo = np.percentile(Y, 1)
#y_hi = np.percentile(Y, 99)
#print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
#Y = np.clip(Y, y_lo, y_hi)

In [ ]:
train_ratio = 0.8
train_size = int(X.shape[1] * train_ratio)

X_train = X[:,:train_size].T
Y_train = Y[:,:train_size].T

X_val = X[:,train_size:].T
Y_val = Y[:,train_size:].T

print(X_train.shape)
print(X_val.shape)

In [ ]:
# Normalize features
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std

# Normalize Y — without this the output layer fights a non-zero offset
# and the model collapses to predicting the mean/median
#y_mean = Y_train.mean()
#y_std  = Y_train.std() + 1e-8
#Y_train_norm = (Y_train - y_mean) / y_std
#Y_val_norm   = (Y_val   - y_mean) / y_std
#print(f"Y normalized: mean→0, std→1  (y_mean={y_mean:.4f}, y_std={y_std:.4f})")


In [ ]:
model.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=100,
                  verbose=1,
                  shuffle=True)


In [ ]:
import matplotlib.pyplot as plt

def plot_acc_and_loss(history, title):
    plt.title("Model and Validation MAE for " + title)
    xs = list(range(1, len(history.history['mae']) + 1))
    plt.plot(xs, history.history['mae'], label="Model MAE", color="Red")
    plt.plot(xs, history.history['val_mae'], label="Validation MAE", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()
    plt.show()

    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    plt.show()

def prcnt_within_tolerance(x, y, model, tolerance):
    print("Count:",np.count_nonzero(abs(model.predict(x) - y) <= tolerance))
    print("X Len:", x.shape[0])
    return np.count_nonzero(abs(model.predict(x) - y) <= tolerance) / x.shape[0]

In [ ]:
plot_acc_and_loss(model.history, "Model MAE")

In [ ]:
print("Prediction:", model.predict(X_train[:15]))
print("Label:", Y_train[:15])

In [ ]:
print("Prediction:", model.predict(X_val[:15]))
print("Label:", Y_val[:15])

In [ ]:
modelMSE = tf.keras.Sequential([
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(505,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(126, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

modelMSE.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelMSE.summary()

In [ ]:
modelMSE.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=100,
                  verbose=1,
                  shuffle=True)

In [ ]:
plot_acc_and_loss(modelMSE.history, "MSE Model")

In [ ]:
print("Prediction:", modelMSE.predict(X_val[:15]))
print("Label:", Y_val[:15])

In [ ]:
print("Hubard Within 5%:", prcnt_within_tolerance(X_val, Y_val, model, 0.05))
print("Hubard Within 10%:", prcnt_within_tolerance(X_val, Y_val, model, 0.1))
print("Hubard Within 20%:", prcnt_within_tolerance(X_val, Y_val, model, 0.2))

In [ ]:
print("MSE Within 5%:", prcnt_within_tolerance(X_val, Y_val, modelMSE, 0.05))
print("MSE Within 10%:", prcnt_within_tolerance(X_val, Y_val, modelMSE, 0.1))
print("MSE Within 20%:", prcnt_within_tolerance(X_val, Y_val, modelMSE, 0.2))

In [ ]:
modelWD = tf.keras.Sequential([
    tf.keras.layers.Dense(1024, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1024, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

modelWD.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelWD.summary()

In [ ]:
modelWD.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=100,
                  verbose=1,
                  shuffle=True)

In [ ]:
modelLSTM = tf.keras.Sequential([
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Reshape((252,1)),
    tf.keras.layers.LSTM(units=10, input_shape=(252,), return_sequences=False),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

modelLSTM.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanAbsoluteError(),
    metrics=['mae']
)

modelLSTM.summary()

In [ ]:
print("WD Within 5%:", prcnt_within_tolerance(X_val, Y_val, modelWD, 0.05))
print("WD Within 10%:", prcnt_within_tolerance(X_val, Y_val, modelWD, 0.1))
print("WD Within 20%:", prcnt_within_tolerance(X_val, Y_val, modelWD, 0.2))

In [ ]:
plot_acc_and_loss(modelMSE.history, "WD Model")

In [ ]:
modelLSTM.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=100,
                  verbose=1,
                  shuffle=True)
